
# Atari Pong (APong) with PyAOgmaNeo

This notebook walks through training **PyAOgmaNeo** to play Atari Pong directly from pixels.

In this tutorial, we will:

- install and imports everything it needs,
- explain how the environment and agent are set up,
- show the full training loop,
- evaluate the final agent, and
- point to how this compares to the classic Deep Q‑Network (DQN) Atari results.

You can read it top‑to‑bottom or jump to the sections you're interested in.



## 0. Dependencies

PyAOgmaNeo relies on **Gymnasium** and **ALE** for Atari environments, plus **OpenCV** for image pre‑processing.

If you are running this on a fresh environment, uncomment and run the cell below once to install:

- `gymnasium[atari,accept-rom-license]`
- `ale-py`
- `opencv-python`
- `pyaogmaneo`


In [ ]:

# install dependencies into this environment.
# Run this once.
# %pip install "gymnasium[atari,accept-rom-license]" ale_py opencv-python pyaogmaneo



## 1. Setup and imports

This notebook assumes a standard Python scientific stack plus:

- **Gymnasium** with the Atari extension and ALE,
- **PyAOgmaNeo** for Sparse Predictive Hierarchies (SPH),
- **OpenCV** for image pre-processing, and
- **Matplotlib** for simple plots.

The cell below imports everything and registers the Atari environments with Gymnasium.


In [ ]:

import os
import time
from copy import copy
import csv

import numpy as np
import gymnasium as gym
import ale_py
import cv2

import pyaogmaneo as neo

import matplotlib.pyplot as plt

# Register Atari environments with Gymnasium
gym.register_envs(ale_py)

print("Libraries imported.")



## 2. Configuration

Here we collect all of the important hyperparameters in one place:

- **Visual pre-processing**: how big the down-sampled Pong frames should be, and how much of the score / border area to crop away.
- **SPH architecture**: the size of the ImageEncoder and the hierarchy layer that will learn from the visual features.
- **Training schedule**: number of episodes, maximum steps per episode, exploration rate, and how often to save models.
- **CPU threads**: how many threads PyAOgmaNeo is allowed to use.

Feel free to tweak these, but the defaults are the ones used for the benchmark numbers described in the documentation.


In [ ]:

# 2. Configuration

# Whether to load an existing trained model (pong.ohr / pong.oenc)
load = False

# Visual pre-processing
image_size = (64, 64, 3)     # target (H, W, C)
crop_height_offset = 16      # shift cropping window down to cut off the score area

# SPH architecture
enc_hidden_size = (10, 10, 32)   # ImageEncoder hidden size (width, height, column size)
hierarchy_size = (7, 7, 64)      # Hidden hierarchy layer size
num_layers = 1                   # Number of hierarchy layers

# Training schedule
max_episodes = 1000              # Number of training episodes
max_timesteps = 10_000           # Max steps per episode
exploration_rate = 0.01          # ε-greedy exploration probability
save_frequency = 50              # Save model every N episodes

# Parallelism for PyAOgmaNeo
neo.set_num_threads(4)

print(f"Configured for {max_episodes} training episodes.")
print(f"Image size: {image_size}, encoder: {enc_hidden_size}, hierarchy: {hierarchy_size}")



## 3. Atari Pong environment

We use the `ALE/Pong-v5` environment from **Gymnasium**, which wraps the Arcade Learning Environment (ALE):

- observations are `210×160×3` RGB frames,
- rewards are `-1`, `0`, or `+1` when a point is scored,
- an episode ends when either player reaches 21 points.

The environment internally uses a frame-skip of 4, matching the setup used in many Deep Q-Network (DQN) Atari experiments.


In [ ]:

# 3. Create the Atari Pong environment

env_id = "ALE/Pong-v5"  # requires gymnasium[atari,accept-rom-license]
env = gym.make(env_id)  # add render_mode='human' to watch the game

num_actions = env.action_space.n
obs_shape = env.observation_space.shape

min_size = min(obs_shape[0], obs_shape[1])
max_size = max(obs_shape[0], obs_shape[1])

print(f"Environment: {env_id}")
print(f"Observation shape: {obs_shape}")
print(f"Number of actions: {num_actions}")
print(f"Cropping: {min_size} x {min_size} from {obs_shape[0]} x {obs_shape[1]}")



## 4. Building the APong agent

An APong agent has two main parts:

1. **ImageEncoder** – converts each preprocessed frame (`64×64×3`) into a **Column-Sparse Distributed Representation (CSDR)** of size `10×10×32`. Only a single cell per column is active, which keeps computation cheap.
2. **Hierarchy** – a Sparse Predictive Hierarchy that:
   - receives the visual code as input,
   - predicts what will happen next,
   - and includes a special **action output** that learns to predict which discrete Pong action to take.

Actions are treated as another kind of prediction. During learning, we pass the reward into the hierarchy so that it can align its predictions with actions that lead to better outcomes.


In [ ]:

# 4. Build the APong agent: ImageEncoder + Hierarchy

h = None   # Hierarchy (main predictive model)
enc = None # ImageEncoder (visual pre-encoder)

if load:
    print("Loading existing models from disk...")
    try:
        enc = neo.ImageEncoder(file_name="pong.oenc")
        h = neo.Hierarchy(file_name="pong.ohr")
        print("Models loaded.")
    except Exception as e:
        print("Failed to load models, creating new ones instead.")
        print("Reason:", repr(e))
        load = False

if not load:
    print("Creating new models from scratch...")

    # ImageEncoder: visual feature extractor (dense RGB -> sparse CSDR)
    enc = neo.ImageEncoder(
        enc_hidden_size,
        [
            neo.ImageVisibleLayerDesc(
                (image_size[1], image_size[0], image_size[2]),
                8,  # receptive field radius
            )
        ],
    )

    # Hierarchy layer descriptions
    layer_descs = []
    for _ in range(num_layers):
        ld = neo.LayerDesc()
        ld.hidden_size = hierarchy_size
        layer_descs.append(ld)

    # Two IO streams:
    #  - index 0: visual features (no direct predictions)
    #  - index 1: discrete action predictions
    h = neo.Hierarchy(
        [
            neo.IODesc(size=enc.get_hidden_size(), io_type=neo.none, up_radius=3),
            neo.IODesc(size=(1, 1, num_actions), io_type=neo.action, up_radius=0, down_radius=3),
        ],
        layer_descs,
    )

    # Balance between "regular" prediction and action learning
    h.params.ios[1].importance = 0.5

print("Models ready.")
print("  ImageEncoder hidden size:", enc.get_hidden_size())
print("  Hierarchy hidden size:", h.get_hidden_size(0))
print("  Action space size:", num_actions)



## 5. Visual pre-processing

The raw Atari frames include a score bar and unused borders. To keep the network small and focused, we:

1. crop out a centered square region that excludes the score,
2. shift the crop down a little (`crop_height_offset`) so the paddle and ball stay in view, and
3. resize to `64×64` pixels.

We keep RGB colour channels instead of converting to grayscale; colour gives the encoder extra information to distinguish objects and background even though Pong is visually simple.


In [ ]:

def preprocess_frame(obs):
    # Preprocess a raw Atari frame for the SPH agent:
    # 1. Crop a square region, removing the score area at the top.
    # 2. Resize to the network's input resolution.
    # 3. Keep RGB channels (no grayscale conversion).

    # Crop to a centered square region and shift down to remove the score
    start_row = max_size // 2 - min_size // 2 + crop_height_offset
    end_row = max_size // 2 + min_size // 2 + crop_height_offset
    cropped = obs[start_row:end_row, :, :]

    # Resize to the network input size
    resized = cv2.resize(
        cropped,
        (image_size[0], image_size[1]),
        interpolation=cv2.INTER_LINEAR,
    )

    return resized

# Quick sanity check
test_obs, _ = env.reset()
processed = preprocess_frame(test_obs)
print(f"Original frame: {test_obs.shape} -> Processed: {processed.shape}")



## 6. Training loop (online predictive RL)

PyAOgmaNeo does not learn a Q-function like DQN. Instead, it performs **online predictive learning**:

1. The encoder turns the current frame into a sparse code.
2. The hierarchy takes the sparse code and the previous action as input.
3. It predicts the *next* code and the *next* action.
4. We execute the predicted action (with a bit of ε-greedy exploration).
5. We feed the **observed reward** back into the hierarchy (scaled up) to bias learning toward action sequences that tend to win points.

Because SPH is fully online and highly sparse, we can keep learning every frame without a replay buffer or GPU.

The helper function below logs per-episode rewards and an exponential moving average, and also writes them out to `apong_training_metrics.csv` for later analysis (for example, in R).


In [ ]:

def train_agent(
    num_episodes=max_episodes,
    max_timesteps=max_timesteps,
    exploration_rate=exploration_rate,
    reward_scale=100.0,
    save_every=save_frequency,
):
    # Train the APong agent and log episodic rewards.
    # Returns:
    #   episode_rewards : np.ndarray
    #   ema_rewards     : np.ndarray (exponential moving average)
    #   first_win_episode : int or None (1-based index of first episode with total reward > 0)

    global h, enc

    episode_rewards = []
    ema_rewards = []
    best_reward = -np.inf
    first_win_episode = None

    print(f"Starting training for {num_episodes} episodes...\n")
    action = 0
    prev_reward = 0.0

    for episode in range(num_episodes):
        obs, info = env.reset()
        total_reward = 0.0
        prev_reward = 0.0

        for t in range(max_timesteps):
            # 1) Encode visual input
            processed_obs = preprocess_frame(obs)
            enc.step([processed_obs.ravel()], True)

            # 2) Predict next state + action, and learn from previous reward
            h.step([enc.get_hidden_cis(), [action]], True, prev_reward * reward_scale)

            # 3) Choose next action from predictions
            action = int(h.get_prediction_cis(1)[0])

            # ε-greedy exploration
            if np.random.rand() < exploration_rate:
                action = np.random.randint(0, num_actions)

            # 4) Step the environment
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            prev_reward = reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)
        if episode == 0:
            ema = total_reward
        else:
            ema = 0.9 * ema_rewards[-1] + 0.1 * total_reward
        ema_rewards.append(ema)

        if total_reward > best_reward:
            best_reward = total_reward

        if total_reward > 0 and first_win_episode is None:
            first_win_episode = episode + 1

        print(
            f"Episode {episode + 1:4d}: "
            f"{t + 1:4d} steps, "
            f"reward {total_reward:6.1f}, "
            f"EMA avg {ema:6.2f}"
        )
        if first_win_episode == episode + 1:
            print(f"  --> first positive total reward at episode {first_win_episode}!")

        if (episode + 1) % save_every == 0:
            h.save_to_file("pong.ohr")
            enc.save_to_file("pong.oenc")
            print(f"Saved intermediate models at episode {episode + 1}")

    # Save final models
    h.save_to_file("pong_final.ohr")
    enc.save_to_file("pong_final.oenc")

    # Save metrics to CSV for external tools (e.g. R)
    with open("apong_training_metrics.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["episode", "reward", "ema_reward"])
        for ep, r, ema in zip(range(1, num_episodes + 1), episode_rewards, ema_rewards):
            writer.writerow([ep, r, ema])

    print("\nTraining finished.")
    print(f"Best per-episode reward: {best_reward:.1f}")
    if first_win_episode is not None:
        print(f"First episode with positive total reward: {first_win_episode}")
    else:
        print("Agent did not achieve a positive total reward during this run.")

    return np.array(episode_rewards), np.array(ema_rewards), first_win_episode


# Run training (this may take a while for 1000 episodes)
episode_rewards, ema_rewards, first_win_episode = train_agent()



### 6.1. Quick training curve

The cell below plots the raw reward per episode together with a smoothed exponential moving average, so you can visually check that learning is happening.


In [ ]:

episodes = np.arange(1, len(episode_rewards) + 1)

plt.figure()
plt.plot(episodes, episode_rewards, label="Episode reward")
plt.plot(episodes, ema_rewards, label="EMA reward (0.9/0.1)")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("APong training curve (PyAOgmaNeo)")
plt.legend()
plt.grid(True)
plt.show()



## 7. Evaluation and typical results

After training, we freeze learning and simply let the agent play a few games. The helper function below runs `num_episodes` evaluation games and prints the total reward for each.

In our reference run with this configuration:

- training ran for 1,000 episodes,
- the **first winning game** (total reward > 0) appeared around episode **700**,
- final rewards were consistently positive and comparable to standard deep-RL baselines trained on Pong,
- all of this was achieved **on CPU only**, with no replay buffer and no backpropagation inside the SPH hierarchy.

Your exact numbers will vary with random seeds and hardware, but you should see a clear progression from large negative scores towards positive ones.


In [ ]:

def test_agent(num_episodes=5):
    # Run the trained agent without learning and report total rewards.
    print(f"\nTesting agent for {num_episodes} episodes...\n")

    test_rewards = []
    action = 0

    for episode in range(num_episodes):
        obs, _ = env.reset()
        total_reward = 0.0

        for t in range(max_timesteps):
            processed_obs = preprocess_frame(obs)

            # Forward pass only (no learning)
            enc.step([processed_obs.ravel()], False)
            h.step([enc.get_hidden_cis(), [action]], False, 0.0)

            # Choose the predicted action
            action = int(h.get_prediction_cis(1)[0])

            obs, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward

            if terminated or truncated:
                break

        test_rewards.append(total_reward)
        print(f"Test episode {episode + 1}: total reward {total_reward:.1f} in {t + 1} steps")

    avg_test_reward = float(np.mean(test_rewards))
    print("\nSummary over test episodes:")
    print(f"  Average reward: {avg_test_reward:.2f}")
    print(f"  Best game:      {max(test_rewards):.1f}")
    print(f"  Worst game:     {min(test_rewards):.1f}")

    return test_rewards

# Example: 3 evaluation games
test_results = test_agent(num_episodes=3)



## 9. How this differs from DQN

Classic Deep Q-Network (DQN) agents for Atari:

- turn stacks of down-sampled grayscale frames (`84×84×4`) into Q-values with a deep convolutional network,
- use replay buffers with around a million frames and target networks for stability,
- are typically trained for **tens of millions of frames** per game,
- and are usually run on a GPU.

PyAOgmaNeo / SPH takes a different route:

- learns **predictive sparse codes** rather than dense CNN features,
- updates weights **online** every frame using local learning rules (no backprop through time),
- uses no replay buffer and very small models (on the order of a million *sparse* synapses),
- runs comfortably on a single CPU core (even on Raspberry Pi‑class hardware in the AOgmaNeo C++ demos).

The APong experiments show that this predictive, sparse approach can reach Pong performance comparable to DQN-style baselines while using **far fewer frames and dramatically less compute**.



## 10. Cleanup

Always close the Gymnasium environment when you are done.


In [ ]:

# 10. Cleanup

env.close()
print("Environment closed. Models were saved to disk if training was run.")
